# 04 – Run Agentic Critique
Multi-agent debate loop: Reader → Critic ↔ Auditor (up to N rounds) → Summariser.

Produces structured reviews with `summary`, `strengths`, `weaknesses`, `questions`, `scores`.

In [ ]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

from src.agents.orchestrator import load_config, run_all_papers, run_agentic_critique
import json

cfg = load_config('../config.yaml')

# Show key agent config
print(f"Max rounds:  {cfg['agent']['max_rounds']}")
print(f"Use tools:   {cfg['agent']['use_tools']}")
print(f"Truncate at: {cfg['agent']['truncate_body_chars']} chars")
print(f"Model map:   {cfg['agent']['model_map']}")

In [ ]:
# Run on all papers (skips already-processed ones)
run_all_papers(
    reviews_path='../data/processed/reviews_parsed.json',
    output_dir='../results/agents',
    cfg=cfg,
)

In [ ]:
# Inspect a sample result
from pathlib import Path

results = sorted(Path('../results/agents').glob('*.json'))
results = [r for r in results if r.stem not in ('scores', 'judge_scores')]
print(f'Agent results: {len(results)}')

if results:
    with open(results[0]) as f:
        r = json.load(f)

    print(f"\n--- {r['paper_id']} ({r.get('title', '')[:50]}) ---")
    print(f"Rounds: {r['rounds']}  |  Latency: {r['latency_seconds']}s  |  Tokens: {r['token_usage']}")

    structured = r.get('structured', {})
    print(f"\nSummary: {structured.get('summary', 'N/A')[:200]}")

    print(f"\nStrengths ({len(structured.get('strengths', []))}):")
    for s in structured.get('strengths', [])[:3]:
        print(f"  + {s['point'][:100]}")

    print(f"\nWeaknesses ({len(structured.get('weaknesses', []))}):")
    for w in structured.get('weaknesses', [])[:3]:
        print(f"  - {w['point'][:100]}")

    print(f"\nQuestions ({len(structured.get('questions', []))}):")
    for q in structured.get('questions', [])[:3]:
        print(f"  ? {q['question'][:100]}")

    print(f"\nScores: {structured.get('scores', {})}")

In [ ]:
# Latency and token usage distribution across all papers
import matplotlib.pyplot as plt

if len(results) > 1:
    latencies, tokens_in, tokens_out = [], [], []
    for rf in results:
        with open(rf) as f:
            d = json.load(f)
        latencies.append(d.get('latency_seconds', 0))
        tokens_in.append(d.get('token_usage', {}).get('input', 0))
        tokens_out.append(d.get('token_usage', {}).get('output', 0))

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].hist(latencies, bins=15, edgecolor='white', color='#DD8452')
    axes[0].set_title('Latency per paper (seconds)')
    axes[0].set_xlabel('Seconds')

    axes[1].hist(tokens_in, bins=15, edgecolor='white', color='#4C72B0', alpha=0.7, label='Input')
    axes[1].hist(tokens_out, bins=15, edgecolor='white', color='#DD8452', alpha=0.7, label='Output')
    axes[1].set_title('Token usage per paper')
    axes[1].set_xlabel('Tokens')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    print(f"Mean latency: {sum(latencies)/len(latencies):.1f}s")
    print(f"Mean tokens:  {sum(tokens_in)//len(tokens_in):,} in / {sum(tokens_out)//len(tokens_out):,} out")